In [2]:
#constants
POLICY = 'Pòlissa/Póliza/Policy'
TECHNOLOGY = 'Tecnologia/Tecnología/Technology'
DIAMETER = 'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)'
USAGE = 'Ús/Uso/Use'
HOUSING = "Tipus d'habitatge/Tipo de vivienda/Type of housing"
DATE = 'Data/Fecha/Date'
CONSUMPTION = 'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)'

In [3]:
import pyarrow.dataset as ds

file_path = '../../data/lectures_horaries_ABD.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()

**Dataset Ajustement (a column per hour for every date and policy)**

In [ ]:
import pandas as pd

# Convert 'Data/Fecha/Date' column to datetime format
df['Data/Fecha/Date'] = pd.to_datetime(df['Data/Fecha/Date'])

# Extract the date and hour separately
df['Date'] = df['Data/Fecha/Date'].dt.date
df['Hour'] = df['Data/Fecha/Date'].dt.hour

# Pivot the table so that each hour is a separate column
df_pivot = df.pivot_table(index='Date', columns='Hour', values='Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)')

# Rename columns as hour0, hour1, ..., hour23
df_pivot.columns = [f'hour{int(hour)}' for hour in df_pivot.columns]

# Reset index to make Date a column
df_pivot = df_pivot.reset_index()

# Keep only one row per date
df_deduped = df.drop_duplicates(subset='Date')[['Date', 'Pòlissa/Póliza/Policy', 'Tecnologia/Tecnología/Technology',
                                                'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)',
                                                'Ús/Uso/Use', 'Tipus d\'habitatge/Tipo de vivienda/Type of housing']]

# Merge the pivot table with the deduplicated columns
df_final = pd.merge(df_pivot, df_deduped, on='Date', how='left')

cols = ['Pòlissa/Póliza/Policy', 'Date'] + [col for col in df_final.columns if col not in ['Pòlissa/Póliza/Policy', 'Date']]
df_final = df_final[cols]

df_final.head()

**Drop of rows with at least one null value**

In [ ]:
hour_columns = [f'hour{hour}' for hour in range(24)] # Subset of the hour columns
df_cleaned = df_final.dropna(subset=hour_columns, how='any')

df_cleaned.info()

In [ ]:
print(df.head())  # First 5 rows

In [ ]:
print(df.columns) # Column names

In [ ]:
print(df.info())  # Data types and missing values 